# Task 1 — Step 2: DPO Regularization Strength Study ($\beta$ Sweep)

### Research Question
How does changing the regularization parameter $\beta \in \{0.03, 0.10, 0.30\}$ alter:
1. Preference fitting (held-out preference accuracy)?
2. Reference policy drift ($D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}})$)?
3. Reward model score and generated response length?

Is the Pareto relationship between reward and reference-model drift monotonic over this range?

### Experimental Protocol
- Start from the identical `Qwen2.5-1.5B-Instruct` LoRA initialization.
- Train short forks (600 examples, matched optimizer, seed, and LoRA rank).
- Evaluate on the identical 300 held-out UltraFeedback pairs.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run Beta Sweep or Load Cached Results


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task1_dpo.ablate_beta --config configs/dpo.yaml
else:
    print("[OK] Loading saved beta ablation results...")


## 2. Beta Ablation Comparative Metrics Table


In [ ]:
ablation_file = "results/task1_dpo/dpo_beta_ablation_results.json"
if os.path.exists(ablation_file):
    with open(ablation_file, encoding='utf-8') as f:
        ab_data = json.load(f)
    
    rows = []
    for b_str in sorted(ab_data.keys(), key=float):
        ev = ab_data[b_str].get("eval", {})
        tr = ab_data[b_str].get("train", {})
        rows.append({
            "Beta (β)": float(b_str),
            "Held-out Pref Acc (%)": f"{ev.get('held_out_preference_accuracy', 0)*100:.2f}%",
            "Mean KL Drift": f"{ev.get('mean_kl_from_reference', 0):.4f}",
            "Mean Reward": f"{ev.get('mean_reward', 0):.4f}",
            "Std Reward": f"{ev.get('std_reward', 0):.4f}",
            "Mean Length (words)": f"{ev.get('mean_response_length_words', 0):.1f}",
            "Final Train Loss": f"{tr.get('final_loss', 0):.4f}"
        })
    df_beta = pd.DataFrame(rows).sort_values("Beta (β)")
    display(df_beta)


## 3. Visualizing the Reward vs. KL Pareto Frontier & Beta Dynamics


In [ ]:
if os.path.exists(ablation_file):
    betas = df_beta["Beta (β)"].values
    rewards = [float(r["Mean Reward"]) for _, r in df_beta.iterrows()]
    kls = [float(r["Mean KL Drift"]) for _, r in df_beta.iterrows()]
    accs = [float(r["Held-out Pref Acc (%)"].replace('%','')) for _, r in df_beta.iterrows()]
    lens = [float(r["Mean Length (words)"]) for _, r in df_beta.iterrows()]
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Pareto Frontier (Reward vs KL)
    axes[0].plot(kls, rewards, marker='o', markersize=10, color='#8B5CF6', lw=2.5)
    for i, b in enumerate(betas):
        axes[0].annotate(f"β={b}", (kls[i], rewards[i]), textcoords="offset points", xytext=(8, -8), fontweight='bold')
    axes[0].set_title('Pareto Frontier: Reward vs. Reference Drift')
    axes[0].set_xlabel('Mean KL(π_θ || π_ref)')
    axes[0].set_ylabel('Mean Reward-Model Score')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Preference Accuracy vs Beta
    axes[1].plot(betas, accs, marker='s', markersize=10, color='#EC4899', lw=2.5)
    axes[1].set_title('Held-out Preference Accuracy vs. β')
    axes[1].set_xlabel('Regularization Parameter β')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True, alpha=0.3)
    
    # 3. Generated Length vs Beta
    axes[2].bar([str(b) for b in betas], lens, color='#3B82F6', width=0.4, alpha=0.85)
    for i, l in enumerate(lens):
        axes[2].text(i, l + 1, f"{l:.1f}", ha='center', fontweight='bold')
    axes[2].set_title('Generated Response Length vs. β')
    axes[2].set_xlabel('Regularization Parameter β')
    axes[2].set_ylabel('Mean Words')
    axes[2].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Publication Dashboard Inspection


In [ ]:
for p in ["report/figures/task1_dpo_beta_ablation.png", "report/figures/task1_dpo_reward_vs_kl_frontier.png"]:
    if os.path.exists(p):
        print(f"Displaying publication figure: {p}")
        display(Image(filename=p, width=650))


## 5. Research Findings & Report Notes
- **Monotonicity**: Does increasing $\beta$ monotonically tighten reference constraint or increase preference fitting?
- **Trade-off Interpretation**: How does the scale factor interact with gradient magnitude on the preference log-ratio?
